
Start the runtime locally:

- `blaze run -c opt //experimental/users/canliu/graph_flow:notebook_cpu`

# Environment setup

In [ ]:
import dataclasses
import datetime
import functools
import os
import pdb
import time
from typing import List, Optional
from absl import logging
from collections import defaultdict

import dgf
from dgf.src.data import schema as schema_lib
from dgf.src.io import spanner_graph_metadata as sgm
from dgf.src.io import dataset_loader
from dgf.src.transform import merge
from dgf.src.io import spanner as spanner_io_lib
from dgf.src.io import gf_graph_in_beam as gf_graph_in_beam_lib
from dgf.src.util import test_util

import flax.linen as nn
from frozendict import frozendict
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import tqdm

from google.cloud import spanner
import google.auth as google_auth
from absl.testing import absltest

InMemoryGraph = dgf.data.InMemoryGraph
InMemoryNodeSet = dgf.data.InMemoryNodeSet
InMemoryEdgeSet = dgf.data.InMemoryEdgeSet

%load_ext spanner_graph_notebook

In [ ]:
#@markdown ## SpannerGraph
#@markdown Some of the code is copied from google3/third_party/py/dgf/src/io/spanner_graph.py.
#@markdown We could extend the existing class.

class SpannerGraph:

  _SPANNER_GRAPH_METADATA_JSON_COLUMN_NAME = "PROPERTY_GRAPH_METADATA_JSON"
  _DGF_SCHEMA_ATTRIBUTE_FEATURE_NAME = "__attributes__"
  _SPANNER_GRAPH_ELEMENT_TYPE_NODE = "NODE"
  _SPANNER_GRAPH_ELEMENT_TYPE_EDGE = "EDGE"
  _SPANNER_GRAPH_ELEMENT_PROPERTIES_KEY = "properties"
  _ARRAY_PREFIX = "ARRAY"

  def __init__(
      self,
      project_id: str,
      instance_id: str,
      database_id: str,
      graph_name: str,
      feature_shapes: Optional[dict[str, tuple[int, ...]]] = None,
      feature_semantics: Optional[dict[str, schema_lib.FeatureSemantic]] = None,
      key_mappings: Optional[dict[str, str]] = None,
      ignored_edge_features: list[str] = list()):
    self.project_id = project_id
    self.instance_id = instance_id
    self.database_id = database_id
    self.graph_name = graph_name
    self._spanner_types = {}
    self.feature_formats: dict[str, schema_lib.FeatureFormat] = {}
    self.feature_shapes = feature_shapes if feature_shapes is not None else {}
    self.feature_semantics = (
        feature_semantics if feature_semantics is not None else {}
    )
    self.key_mappings = key_mappings if key_mappings is not None else {}
    self.ignored_edge_features = ignored_edge_features
    self._connect_to_db()
    self._load_metadata()
    self._load_graph_schema()

  def execute_sql(self, query: str):
    with self._database.snapshot() as snapshot:
      results = snapshot.execute_sql(query)
      return results

  def _connect_to_db(self):
    self._client = spanner.Client(project=self.project_id, client_options={"api_endpoint": "spanner.googleapis.com"})
    self._instance = self._client.instance(self.instance_id)
    self._database = self._instance.database(self.database_id)


  def _load_metadata(self) -> sgm.SpannerGraphMetadata:
    """Returns a Dict of Spanner property graph metadata.

    Returns:
      A Dict of Spanner property graph metadata.
    """

    query = """
    SELECT PROPERTY_GRAPH_METADATA_JSON
    FROM INFORMATION_SCHEMA.PROPERTY_GRAPHS
    WHERE property_graph_name=@graph_name;
    """
    with self._database.snapshot() as snapshot:
      graph_metadata_rows = snapshot.execute_sql(query, {"graph_name": self.graph_name}).to_dict_list()

    if not graph_metadata_rows or len(graph_metadata_rows) != 1:
      raise ValueError(
          f"expected exactly 1 property graph with name: {self.graph_id}, "
          f"found {len(graph_metadata_rows)}"
      )

    spanner_graph_metadata_dict = graph_metadata_rows[0][
        "PROPERTY_GRAPH_METADATA_JSON"
    ]

    self._spanner_graph_metadata = sgm.SpannerGraphMetadata.from_dict(spanner_graph_metadata_dict)
    return self._spanner_graph_metadata

  # TODO(canliu): the edge feature schema could be redundant.
  def _spanner_graph_element_features(
      self,
      graph_element_table: sgm.NodeTable | sgm.EdgeTable,
      ignored_features: list[str] = list(),
  ) -> dict[str, schema_lib.FeatureSchema]:
    """Returns a Dict of Spanner property graph element features.

    The features are generated from the property definitions in the graph
    element table.

    Args:
      graph_element_table: The Spanner property graph element table.
    """
    features = {}
    for property_definition in graph_element_table.property_definitions:
      feature_name = property_definition.property_declaration_name
      if feature_name in ignored_features:
        continue

      for (
          property_declaration
      ) in self._spanner_graph_metadata.property_declarations:
        if property_declaration.name == feature_name:

          # Feature Format
          feature_format, is_utf8_string = (
              spanner_io_lib.spanner_type_to_feature_format(
                  property_declaration.type
              )
          )

          # Feature Shape
          if feature_name in self.feature_shapes:
            feature_shape = self.feature_shapes[feature_name]
          elif self._ARRAY_PREFIX in property_declaration.type:
            feature_shape = (None,)
          else:
            feature_shape = ()

          # Feature Semantic
          if feature_name in self.feature_semantics:
            feature_semantic = self.feature_semantics[feature_name]
          else:
            if (
                feature_format.is_float()
                and len(feature_shape) == 1
                and feature_shape[0] is not None
            ):
              feature_semantic = schema_lib.FeatureSemantic.EMBEDDING

            elif feature_format.is_numerical():
              feature_semantic = schema_lib.FeatureSemantic.NUMERICAL
            else:
              feature_semantic = schema_lib.FeatureSemantic.UNKNOWN

          schema_feature_name = self.key_mappings.get(feature_name, feature_name)
          features[schema_feature_name] = schema_lib.FeatureSchema(
              format=feature_format,
              semantic=feature_semantic,
              shape=feature_shape,
              num_categorical_values=None,
              is_utf8_string=is_utf8_string,
          )
          self.feature_formats[schema_feature_name] = feature_format
          ## TODO(tewariy): Is this required?
          self._spanner_types[schema_feature_name] = property_declaration.type
          break
      ## TODO(tewariy): Add checks for features that are not declared in the
      #  property declarations.

    ## TODO(gbm): do we need to set #ID feature?
    # features[gf_graph_in_beam_lib.KEY_ID] = schema_lib.FeatureSchema(
    #     format=schema_lib.FeatureFormat.BYTES,
    #     semantic=schema_lib.FeatureSemantic.PRIMARY_ID,
    #     shape=(),
    #     num_categorical_values=None,
    # )

    logging.info(
        "%d features attached to the %s %s",
        len(features),
        graph_element_table.kind.upper(),
        graph_element_table.name,
    )
    return features


  def _load_graph_schema(self) -> schema_lib.GraphSchema:
    """Returns a DGF schema for the Spanner Graph."""

    node_sets = {}
    edge_sets = {}

    for node_table in self._spanner_graph_metadata.node_tables:
      logging.info("node_table: %s", node_table.name)
      node_sets[node_table.name] = schema_lib.NodeSchema(
          features=self._spanner_graph_element_features(node_table)
      )

    for edge_table in self._spanner_graph_metadata.edge_tables:
      logging.info("edge_table: %s", edge_table.name)
      edge_sets[edge_table.name] = schema_lib.EdgeSchema(
          source=edge_table.source_node_table.node_table_name,
          target=edge_table.destination_node_table.node_table_name,
          features=self._spanner_graph_element_features(
              edge_table,
              ignored_features=self.ignored_edge_features),
      )

    self._graph_schema = schema_lib.GraphSchema(
        node_sets=node_sets,
        edge_sets=edge_sets,
    )
    return self._graph_schema

  @property
  def graph_schema(self):
    return self._graph_schema



In [ ]:
#@markdown ## GraphQueryLanguageGenerator

class GraphQueryLanguageGenerator:
  def __init__(self,
               graph_name: str,
               sampling_config: dgf.sampling.SimpleSamplingConfig):
    self.graph_name = graph_name
    self.sampling_config = sampling_config
    self.seed_ids = []

  def generate(self, seed_ids: list[str], uniformly_random: bool=True) -> str:
    self.seed_ids = seed_ids
    select_graph_query = f"GRAPH {self.graph_name}"

    multi_hops_queries = []
    for hop in range(self.sampling_config.num_hops):
      if hop == 0:
        seed_ids_str = ", ".join([f"'{seed_id}'" for seed_id in seed_ids])
        match_query = f"OPTIONAL MATCH (n{hop}:{self.sampling_config.seed_nodeset})-[e{hop+1}]->(n{hop+1}) WHERE n{hop}.id IN ({seed_ids_str})"
      else:
        match_query = f"OPTIONAL MATCH (n{hop})-[e{hop+1}]->(n{hop+1})"

      # PARTITION BY all nodes in previous hop to ensure in each step, hop_width neighbors are sampled.
      prev_nodes = ", ".join([f"n{prev_hop}" for prev_hop in range(hop+1)])
      maybe_random_sampling = "ORDER BY GENERATE_UUID()" if uniformly_random else ""
      filter_query = f"FILTER IS_FIRST({self.sampling_config.hop_width}) OVER (PARTITION BY {prev_nodes} {maybe_random_sampling})"

      next_nodes = ", ".join(f"n{next_hop}" for next_hop in range(hop+2))
      next_edges = ", ".join([f"e{next_hop}" for next_hop in range(1, hop+2)])

      if hop != self.sampling_config.num_hops - 1:
        intermediate_return_query = f"RETURN {next_nodes}, {next_edges}"
        one_hop_query = "\n".join([match_query, filter_query, intermediate_return_query])
      else:
        one_hop_query = "\n".join([match_query, filter_query])
      multi_hops_queries.append(one_hop_query)

    sample_graph_query = "\n\nNEXT\n\n".join(multi_hops_queries)
    return_query = self._generate_return_query()

    return "\n\n".join([select_graph_query, sample_graph_query, return_query])


  def _all_nodes_fields(self, spanner_field_name: str, struct_field_name: Optional[str] = None) -> str:
    if struct_field_name:
      return ", ".join([f"STRUCT(n{hop}.{spanner_field_name} AS {struct_field_name})" for hop in range(self.sampling_config.num_hops+1)])
    return ", ".join([f"n{hop}.{spanner_field_name}" for hop in range(self.sampling_config.num_hops+1)])


  def _all_edges_fields(self, spanner_field_name: str, struct_field_name: Optional[str] = None) -> str:
    if struct_field_name:
      return ", ".join([f"STRUCT(e{hop}.{spanner_field_name} AS {struct_field_name})" for hop in range(1, self.sampling_config.num_hops+1)])
    return ", ".join([f"e{hop}.{spanner_field_name}" for hop in range(1, self.sampling_config.num_hops+1)])


  def _generate_return_query(self) -> str:
    nodes_to_return = ", ".join([f"TO_JSON(n{hop}) AS n{hop}" for hop in range(self.sampling_config.num_hops+1)])
    edges_to_return = ", ".join(f"TO_JSON(e{hop}) AS e{hop}" for hop in range(1, self.sampling_config.num_hops+1))
    return f"RETURN {nodes_to_return}, {edges_to_return}"

    # TODO(canliu): maybe implement non-JSON format
    # "Nested array is not supported.")
    # node_ids = self._all_nodes_fields("id")
    # # node_features = self._all_nodes_fields(spanner_field_name="feat", struct_field_name="feat")
    # node_features = self._all_nodes_fields("feat")
    # # node_labels = self._all_nodes_fields("label")
    # edge_ids = self._all_edges_fields("id")
    # # edge_target_ids = self._all_edges_fields("targe_id")
    # return_values = [
    #     f"[{node_ids}] AS node_ids",
    #     f"[{node_features}] AS node_features",
    #     # f"[{node_labels}] AS node_labels",
    #     f"[{edge_ids}] AS edge_ids",
    #     # f"[{edge_target_ids}] AS edge_target_ids"
    # ]
    # return f"RETURN {',\n'.join(return_values)}"

In [ ]:
#@markdown ## SpannerGraphSampler

SubgraphFeatures = dict[str, list]
SubgraphFeaturesByGroup = dict[str, SubgraphFeatures]
# seed_id -> {node_set/edge_set -> {feature_name: [features]} }
SubgraphFeaturesBySeed = dict[str, SubgraphFeaturesByGroup]
SubgraphEdgesAdjacencies = dict[str, dict[str, list]]
NodeLookup = dict[str, dict[str, dict[str, int]]]


# TODO(canliu): the current implementation of json_to_in_memory_graphs is not
# fully honoring with_replacement=True.
def json_to_in_memory_graphs(result) -> list[InMemoryGraph]:
  in_memory_graphs_by_seed = defaultdict(InMemoryGraph) # seed_id -> InMemoryGraph
  subgraph_nodes_features = SubgraphFeaturesBySeed()
  subgraph_edges_features = SubgraphFeaturesBySeed()
  # seed_id -> {edge_set -> []}
  subgraph_edges_adjacencies = SubgraphEdgesAdjacencies()
  node_lookup = NodeLookup() # seed_id -> {nodeset_name -> {node_id -> local_index}}

  for row in result:
    seed = row[0]
    seed_id = seed['properties']['id']
    if seed_id not in subgraph_nodes_features:
      # Create a new subgraph rooted in seed_id
      nodeset_name = seed['element_definition_name']
      subgraph_nodes_features[seed_id] = SubgraphFeaturesBySeed()
      subgraph_nodes_features[seed_id][nodeset_name] = SubgraphFeatures()
      for feature_name, val in seed['properties'].items():
        if feature_name == 'id':
          subgraph_nodes_features[seed_id][nodeset_name]['#id'] = [val]
          node_lookup[seed_id] = {nodeset_name: {val: 0}}
        else:
          subgraph_nodes_features[seed_id][nodeset_name][feature_name] = [val]

    for n in range(1, len(row)):
      # An entry could be empty if the sampling has finished.
      if not row[n]:
        continue

      elif row[n]['kind'] == 'edge':
        continue

      elif row[n]['kind'] == 'node':
        neighbor = row[n]
        node_id = neighbor['properties']['id']
        nodeset_name = neighbor['element_definition_name']
        if node_id not in subgraph_nodes_features[seed_id].get(nodeset_name, dict()).get('#id', []):
          for feature_name, val in neighbor['properties'].items():
            if feature_name == 'id':
              node_lookup[seed_id][nodeset_name][val] = len(subgraph_nodes_features[seed_id][nodeset_name]["#id"])
              subgraph_nodes_features[seed_id][nodeset_name]['#id'].append(node_id)
            else:
              subgraph_nodes_features[seed_id][nodeset_name][feature_name].append(val)
      else:
        raise ValueError(f'unrecognized kind: {row[n]["kind"]}')

    for n in range(1, len(row)):
      if not row[n]:
        continue

      elif row[n]['kind'] == 'node':
        continue

      elif row[n]['kind'] == 'edge':
        edge = row[n]
        edgeset_name = edge['element_definition_name']
        # TODO: We can't handle heterogenous graph because we don't know the source and target
        # node sets of the edge. For now, the nodeset_name is hardcoded.
        source_id = node_lookup[seed_id]["nodes"][edge['properties']['id']]
        target_id = node_lookup[seed_id]["nodes"][edge['properties']['target_id']]
        if seed_id not in subgraph_edges_features:
          subgraph_edges_adjacencies[seed_id] = {edgeset_name: []}
          subgraph_edges_features[seed_id] = {edgeset_name: SubgraphFeatures()}

        subgraph_edges_adjacencies[seed_id][edgeset_name].append((source_id, target_id))

        for feature_name, val in edge['properties'].items():
          if feature_name != 'id' and feature_name != 'target_id':
            if feature_name not in subgraph_edge_features[seed_id][edgeset_name]:
              subgraph_edge_features[seed_id][edgeset_name][feature_name] = []
            subgraph_edge_features[seed_id][edgeset_name][feature_name].append(val)
      else:
        raise ValueError(f'unrecognized kind: {row[n]["kind"]}')


  # Convert to InMemoryGraph
  in_memory_graphs = []

  for seed_id in subgraph_nodes_features:
    node_sets = {}
    node_features_array = dict()
    for nodeset_name, features in subgraph_nodes_features[seed_id].items():
      num_nodes = len(features["#id"])
      for feature_name, val in features.items():
        node_features_array[feature_name] = np.array(val)
      node_sets[nodeset_name] = InMemoryNodeSet(num_nodes=num_nodes, features=node_features_array)

    edge_sets = {}
    edge_features_array = dict()
    for edgeset_name in subgraph_edges_adjacencies[seed_id]:
      adjacency = np.array(subgraph_edges_adjacencies[seed_id][edgeset_name]).T
      for feature_name, val in subgraph_edges_features[seed_id].get(edgeset_name, {}).items():
        edge_features_array[feature_name] = np.array(val)
      edge_sets[edgeset_name] = InMemoryEdgeSet(adjacency=adjacency, features=edge_features_array)

    in_memory_graph = InMemoryGraph(node_sets = node_sets, edge_sets = edge_sets)
    in_memory_graphs.append(in_memory_graph)
  return in_memory_graphs

class SpannerGraphSampler:
  def __init__(self,
               graph: SpannerGraph,
               sampling_config: dgf.sampling.SimpleSamplingConfig):
    if sampling_config.reverse:
      raise NotImplementedError("Only reverse=False is supported")

    if not sampling_config.with_replacement:
      raise NotImplementedError("Only with_replacement=True is supported")

    self.graph = graph
    self.query_generator = GraphQueryLanguageGenerator(
        self.graph.graph_name,
        sampling_config)

  def get_query(self, seed_ids: list[str], uniformly_random: bool=True):
    return self.query_generator.generate(seed_ids, uniformly_random)

  def sample_to_json(self, seed_ids: list[str], uniformly_random: bool=True):
    query = self.get_query(seed_ids, uniformly_random)
    return self.graph.execute_sql(query)

  def sample(self, seed_ids: list[str], uniformly_random: bool=True) -> list[InMemoryGraph]:
    json_results = self.sample_to_json(seed_ids, uniformly_random)
    return json_to_in_memory_graphs(json_results)

  def sample_and_merge(self, seed_ids: list[str], uniformly_random: bool=True) -> tuple[InMemoryGraph, dict[str, np.ndarray]]:
    in_memory_graphs = self.sample(seed_ids, uniformly_random)
    return merge.merge_graphs(in_memory_graphs, schema=self.graph.graph_schema, padding=None)

def create_graph_spanner_sampler(graph: SpannerGraph, config: dgf.sampling.SimpleSamplingConfig):
  return SpannerGraphSampler(
      graph=graph,
      sampling_config=config
  )

# Example CUJ

In [ ]:
#@markdown # Global Variables
PROJECT_ID = "biggraphs-poc" #@param{type: "string"}
INSTANCE_ID = "gcp-gnns" #@param {type: "string"}
DATABASE_ID = "ogbn_arxiv" #@param {type: "string"}
GRAPH_NAME = "ogbn_arxiv" #@param {type: "string"}

!gcloud config set project {PROJECT_ID}

#@markdown Run in the local terminal
#@markdown - `gcloud auth login`
#@markdown - `gcloud auth application-default login`

In [ ]:
spanner_graph = SpannerGraph(project_id=PROJECT_ID, instance_id=INSTANCE_ID, database_id=DATABASE_ID, graph_name=GRAPH_NAME,
                             feature_shapes={"feat": (128,)},
                             feature_semantics={"#id": schema_lib.FeatureSemantic.PRIMARY_ID,
                                                "#split": schema_lib.FeatureSemantic.CATEGORICAL,
                                                "labels": schema_lib.FeatureSemantic.CATEGORICAL,
                                                "year": schema_lib.FeatureSemantic.NUMERICAL,
                                                "feat": schema_lib.FeatureSemantic.EMBEDDING},
                             key_mappings={"id": "#id"},
                             ignored_edge_features=["id", "target_id"])

sampling_config = dgf.sampling.SimpleSamplingConfig(
    seed_nodeset="nodes",
    num_hops=2,
    hop_width=5,
    reverse=False,
    with_replacement=True)

sampler = create_graph_spanner_sampler(
    graph=spanner_graph,
    config=sampling_config
)

In [ ]:
# Sample to InMemoryGraph
seed_ids = [10, 11, 12]
subgraphs = sampler.sample(seed_ids, uniformly_random=False)
print(f"length of {len(subgraphs)}")

In [ ]:
# Sample and merge to a single InMemoryGraph
merged_subgraph = sampler.sample_and_merge(seed_ids, uniformly_random=False)

In [ ]:
# Inspect the query
query = sampler.get_query(seed_ids, uniformly_random=False)
print(query)

# In-memory Sampler
- Cross-check the Spanner Graph sampler results with an in-memory-graph sampler

In [ ]:
def compare_topology(g1: InMemoryGraph, g2: InMemoryGraph, allow_isomorphic: bool=True):
  num_nodes_1 = g1.node_sets['nodes'].num_nodes
  num_nodes_2 = g2.node_sets['nodes'].num_nodes
  assert num_nodes_1 == num_nodes_2, f"num_nodes diff {num_nodes_1} != {num_nodes_2}"

  nodes_1 = g1.node_sets['nodes'].features['#id']
  nodes_2 = g2.node_sets['nodes'].features['#id']
  nodes_1 = np.array([int(n[1:]) for n in nodes_1])
  nodes_2 = np.array([int(n) for n in nodes_2])
  print(f"nodes_1: {nodes_1}")
  print(f"nodes_2: {nodes_2}")
  if allow_isomorphic:
    assert set(nodes_1) == set(nodes_2), f"nodes diff {sorted(set(nodes_1))} != {sorted(set(nodes_2))}"
  else:
    assert nodes_1 == nodes_2, f"nodes diff {nodes_1} != {nodes_2}"

  adjacency_1 = g1.edge_sets['edges'].adjacency
  adjacency_2 = g2.edge_sets['edges'].adjacency
  if allow_isomorphic:
    set_adjacency_1, set_adjacency_2 = set(), set()
    for i in range(adjacency_1.shape[1]):
      set_adjacency_1.add((adjacency_1[0][i], adjacency_1[1][i]))
    for i in range(adjacency_2.shape[1]):
      set_adjacency_2.add((adjacency_2[0][i], adjacency_2[1][i]))
    assert set_adjacency_1 == set_adjacency_2, f"adjacency diff: {set_adjacency_1} != {set_adjacency_2}"
  else:
    assert np.array_equal(adjacency_1, adjacency_2), f"adjacency diff: {adjacency_1} != {adjacency_2}"

In [ ]:
in_memory_graph, schema = dataset_loader.load_ogbn_arxiv()

In [ ]:
sampling_plan = dgf.sampling.simple_sampling_config_to_sampling_plan(sampling_config, schema)

print(sampling_plan)

# debug_sampling = not uniformly_random
in_memory_sampler = dgf.sampling.create_sampler(graph=in_memory_graph, plan=sampling_config, schema=schema, batch_size=1, debug_sampling=True)

In [ ]:
in_memory_subgraphs = in_memory_sampler.sample([10, 11, 12])
print(f"length of in-memory subgraphs: {len(in_memory_subgraphs)}")

In [ ]:
idx = 0
compare_topology(in_memory_subgraphs[idx], subgraphs[idx])

In [ ]:
idx = 0

print(f"num_nodes: {subgraphs[idx].node_sets['nodes'].num_nodes}")
print(f"nodes: {subgraphs[idx].node_sets['nodes'].features['#id']}")

print(f"edges: {subgraphs[idx].edge_sets['edges']}")

In [ ]:

print(f"num_nodes: {in_memory_subgraphs[idx].node_sets['nodes'].num_nodes}")
print(f"nodes: {in_memory_subgraphs[idx].node_sets['nodes'].features['#id']}")

print(f"edges: {in_memory_subgraphs[idx].edge_sets['edges']}")

In [ ]:
# Compare in-memory subgraphs and Spanner-Graph subgraphs

class SamplerTest(absltest.TestCase):
  def test_sample(self):
    in_memory_subgraphs = in_memory_sampler.sample([10, 11, 12])
    spanner_subgraphs = sampler.sample(seed_ids=[10, 11, 12], uniformly_random=False)
    test_util.assert_are_equal(self, in_memory_subgraphs, spanner_subgraphs)


In [ ]:
vanilla_tests = SamplerTest()
vanilla_tests.test_sample()

# Notes

## Sampled Spanner Graph Format
- A list where each item is a DFS path from the source, alternating between nodes and edges. For example: `(10), 10->10090, (10090), 10090 -> 142984, (142984)`.
- Each node is a dictionary with keys:
```
{
  'element_definition_name': <>,
  'identifier: <>,
  'kind': <>,
  'labels': <>,
  'properties: {
    'feat': <>,
    'label': <>,
    'id': <>,
    'year': <>
}
```
The fields inside `properties` also depend on the graph's schema.

- Each edge is a dictionary with keys:
```
{
  'destination_node_identifier': <>,
  'element_definition_name': <>,
  'identifier': <>,
  'source_node_identifier': <>,
  'properties': {
    'id': <>,
    'target_id': <>
  }
}
```
Similarly, the fields inside `properties` depend on the graph's schema.